In [1]:
from pathlib import Path
save_dir = Path('/root/autodl-tmp/test_data/')
save_dir.mkdir(parents=True,exist_ok=True)


#### 特征工程
from Data_Pipeline.preprocessors.lob_data_process import process_lob_data
from Data_Pipeline.preprocessors.trade_data_process import process_trade_data
lob_data_dir = '/root/autodl-tmp/ETHUSDT/20levels_parquet'
trade_data_dir = '/root/autodl-tmp/ETHUSDT/trade'


# date = ['2025-11-04','2025-11-05','2025-11-06','2025-11-07','2025-11-08','2025-11-09','2025-11-10',
#         '2025-11-11','2025-11-12','2025-11-13','2025-11-14','2025-11-15','2025-11-16','2025-11-17',
#         '2025-11-18','2025-11-19','2025-11-20','2025-11-21','2025-11-22','2025-11-23','2025-11-24',
#         '2025-11-25','2025-11-26','2025-11-27','2025-11-28','2025-11-29','2025-11-30','2025-12-01',
#         '2025-12-02','2025-12-03','2025-12-04','2025-12-05','2025-12-06','2025-12-07',
#         ]
date  = ['2025-12-08','2025-12-09','2025-12-10','2025-12-11','2025-12-12','2025-12-13',
         '2025-12-14','2025-12-15','2025-12-16','2025-12-17','2025-12-18','2025-12-19',
         '2025-12-20','2025-12-21','2025-12-22','2025-12-23']
levels = 10


lob_data = process_lob_data(data_dir=lob_data_dir,date = date,levels=levels)
trade_data = process_trade_data(data_dir=trade_data_dir,date = date,window_ms=100)

lob_data.write_parquet(save_dir / 'LOB_ETHUSDT_test.parquet')
trade_data.write_parquet(save_dir / 'Trade_ETHUSDT_test_agg100ms.parquet')



libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
import polars as pl
lob_data_path = save_dir / 'LOB_ETHUSDT_test.parquet'
trade_data_path = save_dir / 'Trade_ETHUSDT_test_agg100ms.parquet'
trade_data = pl.read_parquet(trade_data_path)
lob_data = pl.read_parquet(lob_data_path)
from Data_Pipeline.preprocessors.trade_data_process import align_trade_with_lob
trade_data = align_trade_with_lob(trade_data = trade_data,full_lob_data = lob_data)

In [3]:

import numpy as np
from Data_Pipeline.generators.label_gen import generate_data_dict
lob_data_path = save_dir / 'LOB_ETHUSDT_test.parquet'
trade_data_path = save_dir / 'Trade_ETHUSDT_test_agg100ms.parquet'

label_window = 300
levels = 10
change_window = None # none 就是对即时的价格进行预测
need_price = False
need_time = True
data_dict,trade_labels_ret,price_data,time_bucket = generate_data_dict(lob_data_path,
                                                                        trade_data_path,
                                                                        levels=levels,
                                                                        label_window=label_window,
                                                                        need_price=need_price,
                                                                        need_time = need_time)

## 存储为npy格式
np.save(save_dir / 'lob_data.npy',data_dict['lob'])
np.save(save_dir / 'trade_data.npy',data_dict['trade'])
np.save(save_dir / 'trade_labels_ret.npy',trade_labels_ret)
if price_data is not None:
    np.save(save_dir / 'price.npy',price_data)
if time_bucket is not None:
    np.save(save_dir / 'time_bucket.npy',time_bucket)


In [ ]:
import torch
from Model.multi_modal_transformer import MultiModalTransformer
from collections import OrderedDict
## 加载模型
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_version = 'multi_modal_model_3'
checkpoint_path = f'/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}/seed_42/best_model.pt'
model_config_path = '/root/lio/Trade_LOB_MultiModal/checkpoints/multi_modal_model_3/model_config.yaml'
model = MultiModalTransformer.from_config(model_config_path)

checkpoint = torch.load(checkpoint_path, map_location=device,weights_only=False)

# model.to(device)
# model.eval()

# ## 加载数据
# lob_data_test = np.load('/root/autodl-tmp/test_data/lob_data.npy')
# trade_data_test = np.load('/root/autodl-tmp/test_data/trade_data.npy')
# labels_ret_test = np.load('/root/autodl-tmp/test_data/trade_labels_ret.npy')
# time_buckt = np.load('/root/autodl-tmp/test_data/time_bucket.npy')

# print(f"trade_data.shape: {trade_data_test.shape}")
# print(f"lob_data.shape: {lob_data_test.shape}")
# print(f"labels_ret.shape: {labels_ret_test.shape}")

# alpha= 0.001
# labels_class_test = np.ones_like(labels_ret_test, dtype=np.int8)
# # 3. 向量化赋值：涨→2，跌→0
# # 涨：labels_ret > alpha
# labels_class_test[labels_ret_test > alpha] = 2
# # 跌：labels_ret < -alpha
# labels_class_test[labels_ret_test < -alpha] = 0


_IncompatibleKeys(missing_keys=['lob_encoder.conv1.0.weight', 'lob_encoder.conv1.0.bias', 'lob_encoder.conv1.2.weight', 'lob_encoder.conv1.2.bias', 'lob_encoder.conv1.2.running_mean', 'lob_encoder.conv1.2.running_var', 'lob_encoder.conv1.3.weight', 'lob_encoder.conv1.3.bias', 'lob_encoder.conv1.5.weight', 'lob_encoder.conv1.5.bias', 'lob_encoder.conv1.5.running_mean', 'lob_encoder.conv1.5.running_var', 'lob_encoder.conv1.6.weight', 'lob_encoder.conv1.6.bias', 'lob_encoder.conv1.8.weight', 'lob_encoder.conv1.8.bias', 'lob_encoder.conv1.8.running_mean', 'lob_encoder.conv1.8.running_var', 'lob_encoder.conv2.0.weight', 'lob_encoder.conv2.0.bias', 'lob_encoder.conv2.2.weight', 'lob_encoder.conv2.2.bias', 'lob_encoder.conv2.2.running_mean', 'lob_encoder.conv2.2.running_var', 'lob_encoder.conv2.3.weight', 'lob_encoder.conv2.3.bias', 'lob_encoder.conv2.5.weight', 'lob_encoder.conv2.5.bias', 'lob_encoder.conv2.5.running_mean', 'lob_encoder.conv2.5.running_var', 'lob_encoder.conv2.6.weight', 'lo

In [3]:
model

{'epoch': 1,
 'model_state_dict': OrderedDict([('_orig_mod.encoder.conv1.0.weight',
               tensor([[[[-0.3632, -0.5731]]],
               
               
                       [[[-0.1306, -0.1690]]],
               
               
                       [[[ 0.1758, -0.8053]]],
               
               
                       [[[-0.0260, -1.3354]]],
               
               
                       [[[-0.8538, -0.2417]]],
               
               
                       [[[ 0.2631,  0.3929]]],
               
               
                       [[[-0.4039, -0.4044]]],
               
               
                       [[[ 0.2364, -0.3634]]],
               
               
                       [[[ 0.4116,  0.6198]]],
               
               
                       [[[-0.1792, -0.0759]]],
               
               
                       [[[ 0.4344, -0.7913]]],
               
               
                       [[[-0.0264, -0.7936]]],
